# Student Performance Prediction — ML Project
### Stage 2: Data Loading & Exploratory Data Analysis (EDA)

**Author:** Your Name  
**Dataset:** `data/student_performance.csv` (500 students, 6 columns)

---
## What is EDA?
EDA (Exploratory Data Analysis) is the process of **visually and statistically inspecting** your dataset before building any model.

Think of it like a doctor examining a patient before prescribing medicine — you need to understand what you're working with first!

**EDA answers these questions:**
1. What does the data look like? (shape, types)
2. Is there missing or duplicate data? (cleaning needed?)
3. What are the distributions? (are values spread normally or skewed?)
4. Which features are most related to the target? (correlation)
5. Are there outliers? (unusual data points)

---
## Step 1: Import Libraries

Before we do anything, we need to import the tools (libraries) we'll use.

| Library | Purpose |
|---------|----------|
| `pandas` | Load and manipulate the CSV dataset |
| `numpy` | Numerical calculations |
| `matplotlib.pyplot` | Core plotting engine |
| `seaborn` | Beautiful statistical charts built on matplotlib |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This line makes plots appear directly inside the notebook (not in a popup window)
%matplotlib inline

# Set a clean visual style for all seaborn plots
# 'whitegrid' = white background with subtle grid lines
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('Libraries loaded successfully!')

---
## Step 2: Load the Dataset

`pd.read_csv()` reads a CSV file and converts it into a **DataFrame** — 
a table structure with rows (students) and columns (attributes).

In [ ]:
# Load the CSV into a DataFrame
# Make sure you run this from the project root directory!
df = pd.read_csv('../data/student_performance.csv')

print(f'Dataset loaded: {df.shape[0]} students, {df.shape[1]} columns')

---
## Step 3: First Look at the Data

### 3a. `df.head()` — See the First Few Rows

`df.head(n)` shows the first `n` rows. Default is 5.  
This is your **"first handshake"** with the data — does it look reasonable?

In [ ]:
# Show the first 5 rows
df.head()

### 3b. `df.shape` — How Big is the Dataset?

In [ ]:
# df.shape returns (rows, columns)
rows, cols = df.shape
print(f'Rows    : {rows}  (one row = one student record)')
print(f'Columns : {cols}  (5 features + 1 target)')

### 3c. `df.info()` — Column Types and Missing Value Check

`df.info()` is a **health check**. It shows:
- Column names
- How many **non-null** (non-missing) values each column has
- The **data type** of each column (`float64` = decimal, `int64` = whole number)

> ⚠️ If any column shows fewer than 500 non-null values → we have **missing data**!

In [ ]:
df.info()

### 3d. `df.describe()` — Statistical Summary

`df.describe()` gives **8 statistics** for every numeric column:

| Stat | Meaning |
|------|---------|
| `count` | How many non-missing values |
| `mean` | Average value |
| `std` | Standard deviation (how spread out values are) |
| `min` | Smallest value |
| `25%` | 25th percentile (25% of values are below this) |
| `50%` | Median — the exact middle value |
| `75%` | 75th percentile (75% of values are below this) |
| `max` | Largest value |

In [ ]:
df.describe().round(2)

---
## Step 4: Data Quality Checks

### 4a. Missing Values

Missing values (also called **nulls** or **NaN**s) happen when data wasn't collected for a student.  
ML models generally **cannot handle missing values** — we must deal with them first.

In [ ]:
# df.isnull() creates a True/False table: True = missing
# .sum() counts the True values per column
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

if missing.sum() == 0:
    print('\nGreat news: Dataset is complete — no missing values!')
else:
    print('\nAction needed: We have missing values to handle.')

### 4b. Duplicate Rows

A **duplicate row** is when two student records are **completely identical**.  
This is bad because the model might overfit to those repeated examples.

In [ ]:
# df.duplicated() returns True for any row that is an exact copy of a previous row
dup_count = df.duplicated().sum()
print(f'Duplicate rows found: {dup_count}')

if dup_count == 0:
    print('All student records are unique!')
else:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')

---
## Step 5: Visualizations

Now for the fun part — let's **see** the data! Visualizations reveal patterns that numbers alone hide.

### Plot 1: Distribution of Each Feature

A **histogram** shows how often each value appears.  
- **Tall bars** → many students have this value  
- **Short bars** → few students have this value  
- **KDE curve** → the smoothed shape of the distribution  
- **Red dashed line** → the mean (average)

In [ ]:
FEATURES = ['study_hours', 'attendance_pct', 'prev_exam_score',
            'assignments_done', 'sleep_hours']
TARGET   = 'final_score'
all_cols = FEATURES + [TARGET]
colors   = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Distribution of Each Feature & Target Variable',
             fontsize=16, fontweight='bold', y=1.01)

for col, ax, color in zip(all_cols, axes.flatten(), colors):
    sns.histplot(df[col], ax=ax, color=color, kde=True,
                 edgecolor='white', linewidth=0.5)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Number of Students')
    ax.axvline(df[col].mean(), color='red', linestyle='--',
               linewidth=1.5, label=f"Mean: {df[col].mean():.1f}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Plot 2: Correlation Heatmap

**Correlation** measures how strongly two columns move together.  
Range: **-1 to +1**

| Value | Meaning |
|-------|--------|
| +1.0 | Perfect positive relationship |
| 0.0  | No relationship |
| -1.0 | Perfect negative relationship |

**For ML:** High correlation with `final_score` → that feature is probably **very useful** for prediction!

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df.corr(numeric_only=True)

# annot=True: show numbers inside cells | fmt='.2f': 2 decimal places
# cmap='coolwarm': red=high positive, blue=high negative
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Matrix — Features vs Target',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\nCorrelation of each FEATURE with final_score (sorted):')
target_corr = corr_matrix['final_score'].drop('final_score').sort_values(ascending=False)
print(target_corr.round(3).to_string())

### Plot 3: Feature vs Target Scatter Plots

Each dot = one student.  
- **X-axis** = the feature value  
- **Y-axis** = their final score  
- **Color** = how high their score was (yellow = high, purple = low)  
- **Red dashed line** = trend line (fitted using linear regression)

If dots trend **upward left→right**: positive relationship (feature helps predict higher score)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Feature vs Final Score (Scatter Plots with Trend Lines)',
             fontsize=16, fontweight='bold', y=1.01)

for i, (feat, ax) in enumerate(zip(FEATURES, axes.flatten())):
    scatter = ax.scatter(df[feat], df[TARGET],
                         c=df[TARGET], cmap='viridis',
                         alpha=0.5, edgecolors='none', s=20)
    # Trend line using numpy's polyfit
    z = np.polyfit(df[feat], df[TARGET], 1)  # fit a straight line
    p = np.poly1d(z)                          # make it a callable function
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend line')
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel('Final Score')
    ax.set_title(f"{feat.replace('_', ' ').title()} vs Final Score", fontweight='bold')
    ax.legend(fontsize=9)

axes[1, 2].set_axis_off()  # hide the unused 6th subplot
plt.tight_layout()
plt.show()

### Plot 4: Target Variable Distribution

Before modeling, always check: **Is the target variable normally distributed?**  
If it's very skewed, the model might struggle to learn fairly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df[TARGET], kde=True, color='#4C72B0',
             edgecolor='white', linewidth=0.5, ax=ax)
ax.axvline(df[TARGET].mean(),   color='red',   linestyle='--', lw=2,
           label=f"Mean: {df[TARGET].mean():.1f}")
ax.axvline(df[TARGET].median(), color='green', linestyle='--', lw=2,
           label=f"Median: {df[TARGET].median():.1f}")
ax.set_title('Distribution of Final Scores (Target Variable)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Final Score (0-100)')
ax.set_ylabel('Number of Students')
ax.legend()
plt.tight_layout()
plt.show()

### Plot 5: Box Plots — Outlier Detection

A **box plot** is perfect for spotting **outliers** (unusually extreme values).

- **Box** = middle 50% of students (25th to 75th percentile)  
- **Line in box** = median  
- **Whiskers** = extend to min/max within 1.5× the box width  
- **Dots beyond whiskers** = **outliers** ← unusual data points!

In [ ]:
fig, axes = plt.subplots(1, len(all_cols), figsize=(16, 5))
fig.suptitle('Box Plots — Spread and Outlier Detection',
             fontsize=14, fontweight='bold')

for col, ax, color in zip(all_cols, axes, colors):
    sns.boxplot(y=df[col], ax=ax, color=color, width=0.5,
                flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5})
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

---
## Step 6: EDA Summary

Let's consolidate everything we learned from EDA.

In [ ]:
print('=' * 55)
print('  EDA SUMMARY — WHAT WE LEARNED')
print('=' * 55)

print(f'  Dataset shape     : {df.shape}')
print(f'  Missing values    : {df.isnull().sum().sum()} (dataset is clean!)')
print(f'  Duplicate rows    : {df.duplicated().sum()} (all records are unique!)')
print(f'  Target (final_score) range: {df[TARGET].min():.1f} to {df[TARGET].max():.1f}')
print(f'  Target mean       : {df[TARGET].mean():.2f}')
print(f'  Target std dev    : {df[TARGET].std():.2f}')

corr_with_target = df.corr(numeric_only=True)[TARGET].drop(TARGET)
print(f'\n  Feature correlations with final_score:')
for feat, val in corr_with_target.sort_values(ascending=False).items():
    bar = '*' * int(abs(val) * 30)
    print(f'    {feat:<22} r={val:+.3f}  {bar}')

print(f'\n  Strongest predictor : {corr_with_target.idxmax()}')
print(f'  Weakest predictor  : {corr_with_target.idxmin()}')
print('=' * 55)
print('\n  --> Ready for Stage 3: Data Cleaning + Problem Framing!')

---
## Stage 3: Data Cleaning

Even though our dataset is already clean, real-world data is almost always messy.
We will deliberately inject problems into a copy of the data, then clean it step-by-step.

### The 4 Most Common Data Problems

| Problem | What it looks like | How to fix it |
|---------|-------------------|---------------|
| Missing values | `NaN` cells | Fill with median/mode, or drop the row |
| Duplicate rows | Identical student records | `df.drop_duplicates()` |
| Outliers | `study_hours = 99` | Clip to valid range or use IQR |
| Wrong data types | Numbers stored as text | `df['col'].astype(float)` |

In [ ]:
# --- Simulate a DIRTY dataset ---
# We inject 3 types of problems into a copy so we can practice fixing them.
np.random.seed(99)
df_dirty = df.copy()
N = len(df_dirty)

# Problem 1: inject 30 missing values (NaN) across 4 columns
idx = np.random.choice(N, 30, replace=False)
df_dirty.loc[idx[:10],   'study_hours']      = np.nan
df_dirty.loc[idx[10:20], 'attendance_pct']   = np.nan
df_dirty.loc[idx[20:25], 'sleep_hours']      = np.nan
df_dirty.loc[idx[25:],   'assignments_done'] = np.nan

# Problem 2: inject 12 exact duplicate rows
df_dirty = pd.concat([df_dirty, df_dirty.iloc[5:17]], ignore_index=True)

# Problem 3: inject 3 physically impossible outliers
df_dirty.loc[500, 'study_hours']    = 99.0   # impossible
df_dirty.loc[501, 'sleep_hours']    = 0.0    # impossible
df_dirty.loc[502, 'attendance_pct'] = 150.0  # impossible

print('Dirty dataset shape:', df_dirty.shape)
print('\nMissing values per column:')
print(df_dirty.isnull().sum())

### Step A: Fix Missing Values

**Why use MEDIAN (not mean) for continuous columns?**

Mean is pulled by extreme values (outliers). Median is not.
- Values: `[80, 85, 82, 99]` → Mean = 86.5, Median = 82.5
- If 99 is an outlier, median gives a more realistic fill value.

**Why use MODE for discrete/integer columns?**
- Mode = the most frequently occurring value
- Makes sense for whole-number counts like `assignments_done`

In [ ]:
df_fixed = df_dirty.copy()

# Fill CONTINUOUS columns with MEDIAN (robust to outliers)
for col in ['study_hours', 'attendance_pct', 'sleep_hours']:
    median_val = df_fixed[col].median()
    df_fixed[col] = df_fixed[col].fillna(median_val)
    print(f'{col}: filled missing values with median = {median_val:.2f}')

# Fill DISCRETE column with MODE (most common value)
mode_val = df_fixed['assignments_done'].mode()[0]
df_fixed['assignments_done'] = df_fixed['assignments_done'].fillna(mode_val)
print(f'assignments_done: filled missing values with mode = {mode_val}')

print(f'\nMissing values remaining: {df_fixed.isnull().sum().sum()}')

### Step B: Remove Duplicate Rows

In [ ]:
print(f'Shape before: {df_fixed.shape}')

# drop_duplicates() removes rows that are exact copies of earlier rows
# reset_index(drop=True) re-numbers the rows cleanly after removal
df_fixed = df_fixed.drop_duplicates().reset_index(drop=True)

print(f'Shape after removing duplicates: {df_fixed.shape}')

### Step C: Handle Outliers

**Method 1 — IQR (Interquartile Range):**
```
IQR         = Q3 - Q1
Lower fence = Q1 - 1.5 * IQR
Upper fence = Q3 + 1.5 * IQR
```
Values outside these fences are statistical outliers.

**Method 2 — Domain Knowledge (common sense):**
- `study_hours > 24` → physically impossible
- `attendance_pct > 100` → logically impossible
- `sleep_hours = 0` → biologically impossible

We use `df[col].clip(lower, upper)` to cap values at the valid boundary.

In [ ]:
# Domain-knowledge clipping: define min and max for each column
domain_rules = {
    'study_hours':      (0.0, 20.0),
    'attendance_pct':   (0.0, 100.0),
    'sleep_hours':      (1.0, 14.0),
    'assignments_done': (0,   10),
    'prev_exam_score':  (0.0, 100.0),
    'final_score':      (0.0, 100.0),
}

for col, (lo, hi) in domain_rules.items():
    n_out = ((df_fixed[col] < lo) | (df_fixed[col] > hi)).sum()
    if n_out > 0:
        df_fixed[col] = df_fixed[col].clip(lower=lo, upper=hi)
        print(f'{col}: {n_out} outlier(s) clipped to [{lo}, {hi}]')

print(f'\nFinal shape after all cleaning: {df_fixed.shape}')

---
## Stage 3: Problem Framing — Why Regression?

### The Golden Rule

> **Look at your TARGET variable. Ask: "Is it a NUMBER or a CATEGORY?"**
>
> - **Number (continuous)** → **REGRESSION**
> - **Category (label)** → **CLASSIFICATION**

### Our Case

| Question | Answer |
|----------|--------|
| Target variable | `final_score` |
| Sample values | 61.1, 83.2, 84.9, 89.9, 69.0 |
| Type of value | A continuous decimal number between 0 and 100 |
| Problem type | **REGRESSION** |

### What Would Make It Classification Instead?

| If we predicted... | Type |
|--------------------|------|
| Pass / Fail | Binary Classification (2 labels) |
| Grade A / B / C / D / F | Multi-class Classification (5 labels) |
| At-risk: Yes / No | Binary Classification |
| **Actual score (0–100)** | **Regression ← our task** |

### The Analogy
- **Regression** = *"HOW MUCH?"* → "What is the exact score?"
- **Classification** = *"WHICH ONE?"* → "Which grade letter?"

---
## Stage 3: Prepare X and y for Modelling

- **`X`** — the feature matrix: what the model sees as input (500 × 5)
- **`y`** — the target vector: what the model must predict (500 values)

We use the original clean dataset for modelling (not the messy demo version).

In [ ]:
FEATURES = ['study_hours', 'attendance_pct', 'prev_exam_score',
            'assignments_done', 'sleep_hours']
TARGET   = 'final_score'

# X = input features only (we exclude final_score)
# If we included final_score in X, the model would learn nothing useful
# because it would just read the answer directly.
X = df[FEATURES]
y = df[TARGET]

print(f'X shape: {X.shape}  -- rows=students, cols=features')
print(f'y shape: {y.shape}  -- one score per student')
print(f'\nFirst 3 rows of X:')
print(X.head(3))
print(f'\nFirst 3 values of y:')
print(y.head(3).to_string())
print('\nStage 3 complete. Ready for Stage 4: Model Training!')

---
## Stage 4: Model Training

We will train three different ML algorithms and compare their performance.

| Model | Core Idea | Best For |
|-------|-----------|----------|
| Linear Regression | Fit a straight line through data | Simple, linear relationships |
| Decision Tree | Build a tree of yes/no questions | Non-linear patterns, interpretable |
| Random Forest | Average 100 different decision trees | Accuracy + robustness |

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LinearRegression
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

print('Scikit-learn imports complete!')

### Step 1: Train / Test Split

**Why split the data at all?**

If you train and test on the *same* data, the model just memorises the answers —
like re-reading a solved exam paper and acing it again. That tells you nothing
about whether the model *learned* or just *memorised*.

**Solution:** Hide 20% of the data from the model during training.
Only use it at the very end to measure true performance.

```
Full dataset (500 students)
       |
       +------------ 80% = 400 students --> Training set (model learns here)
       |
       +------------ 20% = 100 students --> Test set (sealed envelope)
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,              # feature matrix
    y,              # target vector
    test_size=0.2,  # 20% of data goes to the test set
    random_state=42 # seed: guarantees the same split every time you run this
)

print(f'Training set : {len(X_train)} students (80%) -- model learns from these')
print(f'Test set     : {len(X_test)} students (20%) -- never seen during training')

### Model 1: Linear Regression

**What it does:** Finds the best-fit straight line (or flat plane in 5D) through the data.

It learns one **coefficient** (weight) per feature:
```
final_score = w1*study_hours + w2*attendance_pct + w3*prev_score
            + w4*assignments + w5*sleep_hours + bias
```

**Strength:** Simple, fast, highly interpretable — you can explain exactly why a score changed.

**Weakness:** Assumes relationships are perfectly linear. Cannot capture curves or U-shapes.

In [ ]:
lr_model = LinearRegression()
# .fit() = training step: reads X_train & y_train, finds best-fit coefficients
lr_model.fit(X_train, y_train)
# .predict() = apply the learned line to generate predictions
lr_preds = lr_model.predict(X_test)

print('Learned coefficients (how much each feature moves the score):')
for feat, coef in zip(FEATURES, lr_model.coef_):
    print(f'  {feat:<22}: {coef:+.4f}')
print(f'  bias (intercept)      : {lr_model.intercept_:.4f}')

### Model 2: Decision Tree Regressor

**What it does:** Builds a tree of yes/no questions to narrow down a prediction.

```
Is study_hours > 6?
   YES --> Is attendance_pct > 80?
              YES --> Predict 88.5
              NO  --> Predict 76.2
   NO  --> Is prev_exam_score > 70?
              YES --> Predict 68.4
              NO  --> Predict 52.1
```

**Strength:** Captures non-linear patterns. Easy to visualise.

**Weakness:** Without `max_depth`, it memorises training data (overfitting).
`max_depth=10` limits tree to 10 levels, improving generalisation.

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

print(f'Tree depth     : {dt_model.get_depth()} levels')
print(f'Number of leaves: {dt_model.get_n_leaves()} (each leaf = a final prediction)')

### Model 3: Random Forest Regressor

**What it does:** Builds 100 different Decision Trees, each trained on a random
subset of rows and features. Final prediction = **average of all 100 trees**.

**Analogy:** Instead of consulting one expert, consult 100 different experts
and average their opinions. The crowd is wiser than any individual.

**Why is averaging better?**
Each tree makes different mistakes (random data/features). When averaged,
those mistakes cancel out. This is called **ensemble learning**.

**Strength:** Very accurate, handles non-linearity, resistant to overfitting.

**Weakness:** Slower to train; predictions harder to interpret than a single tree.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,  # build 100 decision trees
    max_depth=10,      # each tree limited to 10 levels (prevents overfitting)
    random_state=42    # reproducibility
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(f'Random Forest trained with {rf_model.n_estimators} trees!')

### Step 2: Evaluate All Three Models

We use **4 standard regression metrics**:

| Metric | Formula | Meaning | Better when |
|--------|---------|---------|-------------|
| **MAE** | avg(|actual - predicted|) | Average error in marks | Lower |
| **MSE** | avg((actual - predicted)²) | Penalises large errors heavily | Lower |
| **RMSE** | sqrt(MSE) | Typical error size, same unit as score | Lower |
| **R²** | 1 - (SS_res / SS_tot) | % of variation explained by the model | Higher |

**Quick intuition for R²:**
- R² = 0.90 → model explains 90% of why scores differ between students
- R² = 0.50 → model explains 50% (a coin flip is accounting for the rest!)
- R² = 1.00 → perfect predictions (unrealistic in practice)

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': round(mae, 4), 'MSE': round(mse, 4),
            'RMSE': round(rmse, 4), 'R2': round(r2, 4)}

results = [
    evaluate_model('Linear Regression', y_test, lr_preds),
    evaluate_model('Decision Tree',     y_test, dt_preds),
    evaluate_model('Random Forest',     y_test, rf_preds),
]

results_df = pd.DataFrame(results).set_index('Model')
print('MODEL COMPARISON TABLE')
print('=' * 60)
print(results_df.to_string())
print('=' * 60)

best_name = results_df['R2'].idxmax()
print(f'\nWinner: {best_name} (highest R2, lowest RMSE)')

### Plot: Metric Comparison Bar Charts

In [ ]:
model_labels = ['Linear\nRegression', 'Decision\nTree', 'Random\nForest']
bar_colors   = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Model Comparison: All 4 Evaluation Metrics', fontsize=14, fontweight='bold')

metric_map = [('MAE','lower=better'), ('MSE','lower=better'),
              ('RMSE','lower=better'), ('R2','higher=better')]

for (metric, label), ax in zip(metric_map, axes):
    vals = [results_df.loc[m.replace('\n',' '), metric] for m in model_labels]
    best_idx = vals.index(min(vals)) if 'lower' in label else vals.index(max(vals))
    bars = ax.bar(model_labels, vals, color=bar_colors,
                  edgecolor='white', linewidth=1.5, width=0.5)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(f'{metric}\n({label})', fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.2)

plt.tight_layout()
plt.show()

### Plot: Actual vs Predicted Scores

Each dot = one test student. X-axis = their real score. Y-axis = model's prediction.

**Perfect model** = all dots on the red diagonal line.
**Scatter around the line** = prediction error. Less scatter = better model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Actual vs Predicted Scores (Test Set)', fontsize=14, fontweight='bold')

for ax, (name, preds, col) in zip(axes,
    [('Linear Regression', lr_preds, '#4C72B0'),
     ('Decision Tree',     dt_preds, '#DD8452'),
     ('Random Forest',     rf_preds, '#55A868')]):
    ax.scatter(y_test, preds, alpha=0.45, color=col, edgecolors='none', s=22)
    lo, hi = min(y_test.min(), preds.min()), max(y_test.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect')
    r2   = results_df.loc[name, 'R2']
    rmse = results_df.loc[name, 'RMSE']
    ax.set_title(f'{name}\nR2={r2:.3f}  RMSE={rmse:.2f}', fontweight='bold')
    ax.set_xlabel('Actual Score')
    ax.set_ylabel('Predicted Score')
    ax.legend()

plt.tight_layout()
plt.show()

### Plot: Random Forest Feature Importance

Random Forest can report which features contributed most to its predictions.
Higher importance = that feature was used more often and reduced error more.

In [ ]:
feat_imp = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(feat_imp['Feature'], feat_imp['Importance'],
               color=bar_colors[:5], edgecolor='white', height=0.55)
for bar, val in zip(bars, feat_imp['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f} ({val*100:.1f}%)', va='center', fontweight='bold')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest: Which Features Matter Most?', fontweight='bold')
ax.set_xlim(0, feat_imp['Importance'].max() * 1.35)
plt.tight_layout()
plt.show()

print('\nFeature importance confirms:')
print(f'  Most important: {feat_imp.iloc[0]["Feature"]} ({feat_imp.iloc[0]["Importance"]:.1%})')
print(f'  Least important: {feat_imp.iloc[-1]["Feature"]} ({feat_imp.iloc[-1]["Importance"]:.1%})')

---
## Stage 5: Best Model Selection & Prediction Function

In Stage 4, **Linear Regression** emerged as the winning model:
- **R² = 0.8941** (explains 89.4% of score variation)
- **RMSE = 4.0941** marks (average prediction error)

Now we will:
1. Save the best model to disk using `pickle` (`src/best_model.pkl`)
2. Build a reusable prediction function with input validation
3. Test it on 5 diverse student profiles

In [ ]:
import pickle
import os

# Save the trained Linear Regression model to disk
os.makedirs('../src', exist_ok=True)
model_path = '../src/best_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(lr_model, f)

print(f'Best model saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path) / 1024:.1f} KB')

### The Prediction Function

This function takes a student's 5 attributes, validates them, formats them
into a DataFrame, calls the model, and clips the result to [0, 100].

In [ ]:
def predict_student_score(study_hours, attendance_pct, prev_exam_score,
                          assignments_done, sleep_hours, model=lr_model):
    """
    Predict final score for a student given their 5 features.
    """
    # 1. Validation
    assert 0 <= study_hours <= 24, 'study_hours must be [0, 24]'
    assert 0 <= attendance_pct <= 100, 'attendance_pct must be [0, 100]'
    assert 0 <= prev_exam_score <= 100, 'prev_exam_score must be [0, 100]'
    assert 0 <= assignments_done <= 10, 'assignments_done must be [0, 10]'
    assert 0 <= sleep_hours <= 24, 'sleep_hours must be [0, 24]'

    # 2. DataFrame input
    input_df = pd.DataFrame([{
        'study_hours':       study_hours,
        'attendance_pct':    attendance_pct,
        'prev_exam_score':   prev_exam_score,
        'assignments_done':  assignments_done,
        'sleep_hours':       sleep_hours,
    }])

    # 3. Predict & Clip
    raw_score = model.predict(input_df)[0]
    final_score = float(np.clip(raw_score, 0.0, 100.0))
    return round(final_score, 1)

print('Prediction function ready!')

In [ ]:
# Test on 5 student profiles
test_profiles = [
    ('Star Performer',       9.0, 95.0, 88.0, 10, 7.5),
    ('Average Student',      5.0, 74.0, 63.0,  5, 7.0),
    ('Struggling Student',   1.5, 52.0, 38.0,  1, 5.0),
    ('Sleep-Deprived Grinder',8.0, 90.0, 80.0, 8, 4.5),
    ('Good Habits, Weak Base',7.0, 88.0, 45.0, 9, 8.0),
]

print(f"{'Profile':<25} {'Study':>6} {'Attend':>7} {'Prev':>6} {'Asgn':>5} {'Sleep':>6} {'Predicted':>10}")
print('-' * 70)
for name, sh, ap, ps, ad, slp in test_profiles:
    sc = predict_student_score(sh, ap, ps, ad, slp)
    print(f"{name:<25} {sh:>6.1f} {ap:>6.1f}% {ps:>6.1f} {ad:>5} {slp:>6.1f} {sc:>9.1f}/100")

---
## End of Notebook — Full Pipeline Complete!

1. **Dataset**: 500 records generated & verified  
2. **EDA**: Visualised distributions, correlations, outliers  
3. **Data Cleaning**: Imputed missing values, removed duplicates, clipped outliers  
4. **Modelling**: Linear Regression (R²=0.894) > Random Forest (0.862) > Decision Tree (0.667)  
5. **Prediction**: Serialised model to disk & built reusable `predict_score()` function  